# Visual Localization

Given a 3D scene reconstructed by the feedforward pipeline, this notebook finds the camera pose of an **external query image** — a frame from a different video — within the scene. Three stages: global frame retrieval via DINOv2-SALAD descriptors, local feature matching with **LoMa** (the dashboard-default matcher), and absolute pose estimation via PnP+RANSAC (pycolmap). XFeat and DISK are drop-in alternatives — see the final section.

In [1]:
%load_ext autoreload
%autoreload 2

## §1 — Load reconstruction

Load the precomputed `FeedforwardResult` from the zarr cache. The reconstruction holds 3D point positions, per-frame camera extrinsics and intrinsics, and the source image paths.

In [ ]:
import os
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pyvista as pv
import torch

%matplotlib inline

%run ../notebook_utils.py
set_notebook_backend()

from collab_splats.pointcloud.feedforward.base import FeedforwardResult
from collab_splats.preproc import FrameStore
from collab_splats.localization import CameraLocalizer, LomaExtractor, estimate_intrinsics, plot_correspondences
from collab_splats.utils.visualization import create_camera_frustum_pyvista, pointcloud_to_polydata

In [ ]:
%run ../tutorial_config.py

# ── Configuration ─────────────────────────────────────────────────────────────
# Reconstruction produced by 02_pointcloud/feedforward_methods.ipynb
assert RECON.exists(), f"missing {RECON} — run 02_pointcloud/feedforward_methods.ipynb first"
print(f"RECON: {RECON}")

In [4]:
# Load the pre-built reconstruction from the zarr cache
result = FeedforwardResult.load_zarr(RECON)
print(f"points: {result.points.shape}  frames: {result.extrinsics.shape[0]}")

points: (500000, 3)  frames: 30


## §2 — Load external query

Load the query image (a frame from a different video, not part of the reconstruction) and estimate its intrinsics — they are unknown, so `estimate_intrinsics` predicts a calibration matrix that PnP later refines. Reference pixels are read from the canonical `frames.zarr` store.

In [ ]:
# External query: a frame from a DIFFERENT video, not part of the reconstruction.
query_image = cv2.cvtColor(cv2.imread(str(QUERY_IMAGE)), cv2.COLOR_BGR2RGB)

# Estimate query intrinsics (experimental; refined by pycolmap focal refinement during PnP).
query_K = estimate_intrinsics(query_image)
print(f"query: {QUERY_IMAGE.name}  shape={query_image.shape}  fx≈{query_K[0, 0]:.0f}")

# Reference pixels come from the canonical frames.zarr (nb 01)
frame_store = FrameStore.open(FRAMES_ZARR)

## §3 — Localize query image

`CameraLocalizer.from_feedforward` indexes the reference frames. `localize` runs global retrieval → **LoMa** matching → PnP+RANSAC and returns a `LocalizationResult` with the estimated world-to-camera pose.

> **Note:** Index build extracts local features for every reference frame — a one-time cost per scene. LoMa weights auto-download on first use (~723 MB).

In [ ]:
# Build the localizer — indexes reference descriptors once (LoMa is the dashboard-default matcher).
# The external query is not in the map, so localize against the FULL reconstruction (no trim).
ref_images = (frame_store.image_by_frame_idx(FrameStore.frame_idx_from_path(p)) for p in result.image_paths)
ref_ids = [Path(p).name for p in result.image_paths]
localizer = CameraLocalizer.from_feedforward(result, images=ref_images, ids=ref_ids, extractor=LomaExtractor())

# Run localization: retrieval → LoMa matching → PnP+RANSAC
loc = localizer.localize(query_image, query_K)
print(
    f"correspondences: {loc.n_correspondences}  inliers: {loc.n_inliers}  pose: {'found' if loc.pose is not None else 'FAILED'}"
)
if loc.pose is None:
    raise RuntimeError("Localization failed — check n_correspondences above.")

## §4 — Correspondences

Visualize the 2D keypoint matches between the query image and the top retrieved reference frame. Inlier matches (green) survive RANSAC; outliers (red) are rejected.

In [ ]:
ranked = loc.ranked_ref_frames
if ranked:
    ri = ranked[0]
    fi = FrameStore.frame_idx_from_path(result.image_paths[ri])
    ref_image = frame_store.image_by_frame_idx(fi)
    plot_correspondences(loc, query_image, ref_image, ref_idx=ri)

## §5 — 3D scene view

Render the reference camera frustums (grey) and the estimated query pose (red) in 3D. The query frustum should be visually consistent with the nearby reference frames.

In [ ]:
# Add point cloud and reference camera frustums
pl = pv.Plotter()
pl.add_mesh(pointcloud_to_polydata(result.points), point_size=2, render_points_as_spheres=True)
for ext in result.extrinsics:
    pl.add_mesh(create_camera_frustum_pyvista(np.linalg.inv(ext), scale=0.05), color="grey", line_width=1)

# Add localized query pose in red
pl.add_mesh(create_camera_frustum_pyvista(np.linalg.inv(loc.pose), scale=0.05), color="red", line_width=3)
pl.add_axes()
pl.show()

## Note on accuracy

The query is an external frame from a different video, so there is no in-map ground-truth pose to compare against. Success is judged by the RANSAC inlier count and by the visual overlay above — the red query frustum should sit consistently among the grey reference frustums.

## Other matchers

Swap the extractor to change the local-matching frontend — retrieval, PnP, and everything else are unchanged:

- `XFeatExtractor()` — XFeat + LightGlue; lighter and faster, fewer correspondences
- `DiskExtractor()` — DISK + LightGlue
- `LomaGExtractor()` — LoMa-G, highest accuracy (larger weights, ~1.4 GB)

```python
from collab_splats.localization import XFeatExtractor

ref_images = (frame_store.image_by_frame_idx(FrameStore.frame_idx_from_path(p)) for p in result.image_paths)
ref_ids = [Path(p).name for p in result.image_paths]
localizer = CameraLocalizer.from_feedforward(result, images=ref_images, ids=ref_ids, extractor=XFeatExtractor())
loc = localizer.localize(query_image, query_K)
```